# 04_03: Text-to-Image Generation

**Objective:** Learn to generate images from text prompts using Stable Diffusion and the HuggingFace `diffusers` library.

**Prerequisites:** Familiarity with HuggingFace ecosystem (Notebook 00_03). GPU strongly recommended for this notebook.

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min VRAM | Recommended Setup | Notes |
|-------------|------------|------|----------|-------------------|-------|
| **Small (GPU)** | stabilityai/sd-turbo | ~5GB | 4GB VRAM | Any GPU | Fast, 1-step generation |
| **Medium (GPU)** | runwayml/stable-diffusion-v1-5 | ~5GB | 6GB VRAM | RTX 4080+ | Classic SD 1.5 |
| **Large (GPU)** | stabilityai/stable-diffusion-xl-base-1.0 | ~7GB | 8GB VRAM | RTX 4080+ | Highest quality |

> **Note:** Text-to-image generation is GPU-intensive. CPU generation is possible but very slow (minutes per image). This notebook includes CPU fallbacks but GPU is strongly recommended.

## Expected Behaviors

### First Time Running
- **Model download**: 3-7GB depending on model choice (can take 5-15 minutes)
- Subsequent runs use cached model files

### What You'll See
- Text prompts are converted into 512x512 or 1024x1024 images
- Same prompt with different seeds produces different images
- Negative prompts help exclude unwanted elements
- More inference steps generally produce higher quality images

### Common Observations
- Specific, descriptive prompts yield better results than vague ones
- Style keywords ("oil painting", "photorealistic", "anime") strongly influence output
- Hands, text, and complex spatial arrangements are common failure modes

## Overview

**Stable Diffusion** generates images through an iterative **denoising** process:

1. Start with random noise
2. A text encoder converts the prompt into a conditioning vector
3. A U-Net denoiser gradually removes noise, guided by the text conditioning
4. After N steps, the denoised latent is decoded into a pixel image

### Architecture

```
Text Prompt → CLIP Text Encoder → Text Embeddings
                                        ↓
Random Noise → U-Net Denoiser (N steps, guided by text) → Denoised Latents
                                                                ↓
                                                     VAE Decoder → Image
```

The entire process happens in **latent space** (compressed representation), which makes it much faster than generating pixels directly.

## Setup and Installation

This notebook requires the `diffusers` library in addition to the standard HuggingFace stack.

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Check for diffusers
try:
    import diffusers
    from diffusers import (
        StableDiffusionPipeline,
        DPMSolverMultistepScheduler,
        EulerDiscreteScheduler,
        EulerAncestralDiscreteScheduler,
    )
    DIFFUSERS_AVAILABLE = True
    print(f'diffusers: {diffusers.__version__}')
except ImportError:
    DIFFUSERS_AVAILABLE = False
    print('diffusers not installed. Install with: pip install diffusers[torch]')
    print('This notebook requires diffusers to generate images.')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if device.type == 'cpu':
    print('\n⚠ WARNING: Text-to-image generation on CPU is very slow (5-15 min per image).')
    print('GPU is strongly recommended for this notebook.')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Model Selection

We use Stable Diffusion v1.5 as the default — it's well-documented, widely used, and produces good results. SD-Turbo is faster (1 step) but lower quality. SDXL produces the highest quality but needs more VRAM.

In [ ]:
# Option 1: Classic Stable Diffusion 1.5 (default, good balance)
MODEL_NAME = 'runwayml/stable-diffusion-v1-5'

# Option 2: SD-Turbo (fast, 1-4 steps, lower quality)
# MODEL_NAME = 'stabilityai/sd-turbo'

# Option 3: SDXL (highest quality, needs 8GB+ VRAM)
# MODEL_NAME = 'stabilityai/stable-diffusion-xl-base-1.0'

# Generation settings
NUM_INFERENCE_STEPS = 25   # More steps = better quality, slower
GUIDANCE_SCALE = 7.5       # How closely to follow the prompt (7-12 typical)
IMAGE_SIZE = 512           # 512 for SD 1.5, 1024 for SDXL

## Loading the Model

Stable Diffusion models are large (~5GB). The first load downloads the model; subsequent loads use the cache. We use `float16` precision on GPU to reduce VRAM usage.

In [ ]:
if DIFFUSERS_AVAILABLE:
    print(f'Loading {MODEL_NAME}...')
    print('This may take several minutes on first run (downloading ~5GB).')
    
    if device.type == 'cuda':
        sd_pipeline = StableDiffusionPipeline.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            safety_checker=None,  # Disable for tutorial speed
        ).to(device)
        # Enable memory-efficient attention if available
        try:
            sd_pipeline.enable_attention_slicing()
        except Exception:
            pass
    else:
        sd_pipeline = StableDiffusionPipeline.from_pretrained(
            MODEL_NAME,
            safety_checker=None,
        ).to(device)
    
    print(f'Model loaded on {device}')
    print(f'Default scheduler: {sd_pipeline.scheduler.__class__.__name__}')
else:
    sd_pipeline = None
    print('Skipping model load — diffusers not installed.')

## Basic Image Generation

Let's start with simple prompts to understand how text descriptions are converted into images. The generator uses a random seed for reproducibility.

In [ ]:
def generate_image(
    prompt: str,
    negative_prompt: str = '',
    num_steps: int = NUM_INFERENCE_STEPS,
    guidance_scale: float = GUIDANCE_SCALE,
    seed: int = SEED,
    width: int = IMAGE_SIZE,
    height: int = IMAGE_SIZE,
) -> Image.Image | None:
    """Generate an image from a text prompt.

    Args:
        prompt: Text description of the desired image.
        negative_prompt: Text describing what to avoid.
        num_steps: Number of denoising steps.
        guidance_scale: How strongly to follow the prompt.
        seed: Random seed for reproducibility.
        width: Image width in pixels.
        height: Image height in pixels.

    Returns:
        Generated PIL Image, or None if diffusers unavailable.
    """
    if sd_pipeline is None:
        print('diffusers not available. Cannot generate image.')
        return None
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    result = sd_pipeline(
        prompt=prompt,
        negative_prompt=negative_prompt if negative_prompt else None,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        width=width,
        height=height,
    )
    
    return result.images[0]


# Generate a simple image
prompt = 'A serene lake surrounded by mountains at sunset, photorealistic'
print(f'Prompt: "{prompt}"')
print(f'Steps: {NUM_INFERENCE_STEPS}, Guidance: {GUIDANCE_SCALE}')

start = time.perf_counter()
image = generate_image(prompt)
elapsed = time.perf_counter() - start

if image:
    print(f'Generated in {elapsed:.1f}s')
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.title(f'{prompt[:50]}...', fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Prompt Engineering

The quality of generated images depends heavily on how you write the prompt. Good prompts are specific, include style descriptors, and mention quality modifiers. Let's compare different prompting strategies.

In [ ]:
# Compare basic vs detailed prompts
prompt_comparison = [
    ('Basic', 'A cat'),
    ('Descriptive', 'A fluffy orange tabby cat sitting on a windowsill'),
    ('Styled', 'A fluffy orange tabby cat, oil painting style, warm lighting, detailed fur'),
    ('Quality+', 'A fluffy orange tabby cat, masterpiece, highly detailed, 4k, professional photography, warm golden hour lighting'),
]

if sd_pipeline is not None:
    fig, axes = plt.subplots(1, len(prompt_comparison), figsize=(5 * len(prompt_comparison), 5))
    
    for idx, (label, prompt) in enumerate(prompt_comparison):
        img = generate_image(prompt, seed=SEED)
        if img:
            axes[idx].imshow(img)
            axes[idx].set_title(f'{label}\n{prompt[:40]}...', fontsize=9)
            axes[idx].axis('off')
    
    plt.suptitle('Effect of Prompt Detail on Image Quality', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Cannot demonstrate prompt engineering without diffusers.')
    print('\nPrompt engineering tips:')
    for label, prompt in prompt_comparison:
        print(f'  [{label}]: {prompt}')

### Prompt Engineering Tips

| Category | Examples | Effect |
|----------|---------|--------|
| **Subject** | "a cat", "a castle", "an astronaut" | Defines what to generate |
| **Style** | "oil painting", "anime", "photorealistic" | Controls artistic style |
| **Quality** | "masterpiece", "highly detailed", "4k" | Improves detail/resolution |
| **Lighting** | "golden hour", "dramatic", "soft" | Controls mood/atmosphere |
| **Camera** | "close-up", "wide angle", "macro" | Controls perspective |
| **Artist** | "in the style of Monet", "by Greg Rutkowski" | Influences aesthetic |

## Negative Prompts

Negative prompts tell the model what to **avoid** generating. They're essential for improving quality by suppressing common artifacts.

In [ ]:
# Demonstrate negative prompts
prompt = 'A portrait of a woman in a garden, detailed, professional'
negative_prompts = [
    '',  # No negative prompt
    'blurry, low quality, distorted, ugly, bad anatomy, bad hands',
]

if sd_pipeline is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    for idx, neg in enumerate(negative_prompts):
        img = generate_image(prompt, negative_prompt=neg, seed=SEED)
        if img:
            axes[idx].imshow(img)
            label = 'No negative prompt' if not neg else f'Negative: {neg[:40]}...'
            axes[idx].set_title(label, fontsize=10)
            axes[idx].axis('off')
    
    plt.suptitle(f'Prompt: "{prompt[:50]}..."', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Common negative prompt terms:')
    print('  blurry, low quality, distorted, ugly, bad anatomy, bad hands,')
    print('  watermark, text, signature, cropped, out of frame')

## Schedulers (Samplers)

The **scheduler** controls the denoising process — how quickly and in what pattern noise is removed over the inference steps. Different schedulers produce different results and have different speed/quality tradeoffs.

In [ ]:
def compare_schedulers(
    prompt: str,
    scheduler_configs: list[tuple[str, object]],
    num_steps: int = 25,
) -> pd.DataFrame:
    """Compare different schedulers on the same prompt.

    Args:
        prompt: Text prompt.
        scheduler_configs: List of (name, scheduler_class) tuples.
        num_steps: Number of inference steps.

    Returns:
        DataFrame with scheduler comparison results.
    """
    if sd_pipeline is None:
        print('diffusers not available.')
        return pd.DataFrame()
    
    original_scheduler = sd_pipeline.scheduler
    results: list[dict[str, str]] = []
    images_list: list[Image.Image] = []
    
    for name, scheduler_class in scheduler_configs:
        sd_pipeline.scheduler = scheduler_class.from_config(sd_pipeline.scheduler.config)
        
        start = time.perf_counter()
        img = generate_image(prompt, num_steps=num_steps)
        elapsed = time.perf_counter() - start
        
        if img:
            images_list.append(img)
            results.append({
                'Scheduler': name,
                'Time (s)': f'{elapsed:.1f}',
                'Steps': str(num_steps),
            })
    
    # Restore original scheduler
    sd_pipeline.scheduler = original_scheduler
    
    # Display images
    if images_list:
        fig, axes = plt.subplots(1, len(images_list), figsize=(5 * len(images_list), 5))
        if len(images_list) == 1:
            axes = [axes]
        for idx, (img, result) in enumerate(zip(images_list, results)):
            axes[idx].imshow(img)
            axes[idx].set_title(f'{result["Scheduler"]}\n({result["Time (s)"]}s)', fontsize=10)
            axes[idx].axis('off')
        plt.suptitle(f'Scheduler Comparison: "{prompt[:40]}..."', fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()
    
    return pd.DataFrame(results)


if DIFFUSERS_AVAILABLE:
    scheduler_configs = [
        ('DPM++ 2M', DPMSolverMultistepScheduler),
        ('Euler', EulerDiscreteScheduler),
        ('Euler Ancestral', EulerAncestralDiscreteScheduler),
    ]
    compare_schedulers('A majestic lion in the savanna, golden light', scheduler_configs)
else:
    print('Scheduler comparison requires diffusers library.')
    print('\nCommon schedulers:')
    print('  - DPM++ 2M: Fast, good quality (recommended default)')
    print('  - Euler: Simple, reliable')
    print('  - Euler Ancestral: More creative/varied outputs')

### Scheduler Selection Guide

| Scheduler | Speed | Quality | Best For |
|-----------|-------|---------|----------|
| **DPM++ 2M** | Fast | High | Default choice, 20-25 steps |
| **Euler** | Fast | Good | Simple, predictable results |
| **Euler Ancestral** | Fast | Good | More creative/varied outputs |
| **DDIM** | Medium | Good | Deterministic, image-to-image |
| **PNDM** | Medium | High | Original SD default |

## Seed Variation

The random seed determines the initial noise pattern, which dramatically affects the output image. Same prompt + same seed = same image (reproducibility). Different seeds explore the model's creative space.

In [ ]:
# Generate variations with different seeds
variation_prompt = 'A cozy cabin in snowy mountains, warm light from windows, winter night'
seeds = [42, 1103, 2024, 7777]

if sd_pipeline is not None:
    fig, axes = plt.subplots(1, len(seeds), figsize=(5 * len(seeds), 5))
    
    for idx, seed in enumerate(seeds):
        img = generate_image(variation_prompt, seed=seed, num_steps=20)
        if img:
            axes[idx].imshow(img)
            axes[idx].set_title(f'Seed: {seed}', fontsize=11)
            axes[idx].axis('off')
    
    plt.suptitle(f'"{variation_prompt[:50]}..."', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Seed variation demo requires diffusers.')
    print(f'Prompt: "{variation_prompt}"')
    print(f'Seeds to try: {seeds}')

## Performance Benchmarking

Generation time depends on the number of inference steps, image resolution, and hardware.

In [ ]:
def benchmark_generation(
    prompt: str,
    step_counts: list[int],
    num_runs: int = 2,
) -> pd.DataFrame:
    """Benchmark image generation speed at different step counts.

    Args:
        prompt: Text prompt.
        step_counts: List of inference step counts to test.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results.
    """
    if sd_pipeline is None:
        print('diffusers not available.')
        return pd.DataFrame()
    
    results: list[dict[str, str]] = []
    for steps in step_counts:
        times: list[float] = []
        for _ in range(num_runs):
            start = time.perf_counter()
            generate_image(prompt, num_steps=steps)
            times.append(time.perf_counter() - start)
        
        results.append({
            'Steps': str(steps),
            'Mean Time (s)': f'{np.mean(times):.1f}',
            'Resolution': f'{IMAGE_SIZE}x{IMAGE_SIZE}',
            'Device': str(device),
        })
    
    return pd.DataFrame(results)


print('=== Generation Speed vs Steps ===')
benchmark_generation(
    'A beautiful landscape, detailed',
    [10, 20, 30, 50],
)

## Exercises

1. **Style transfer via prompts**: Generate the same subject (e.g., "a castle") in different artistic styles: photorealistic, watercolor, pixel art, pencil sketch. Compare the outputs.

2. **Optimal steps**: Generate the same prompt at 5, 10, 15, 20, 30, and 50 steps. At what point do you stop noticing quality improvements?

3. **Guidance scale sweep**: Fix a prompt and vary guidance_scale from 1 to 20. Low values give more creative/random results; high values follow the prompt more strictly but may look oversaturated.

4. **SDXL upgrade**: If you have 8GB+ VRAM, load `stabilityai/stable-diffusion-xl-base-1.0` and compare output quality with SD 1.5 on the same prompts.

## Key Takeaways

- **Stable Diffusion** generates images by iteratively denoising random noise, guided by text embeddings from CLIP
- **Prompt engineering** is crucial: specific subjects + style keywords + quality modifiers produce the best results
- **Negative prompts** suppress common artifacts (blurry, low quality, bad anatomy)
- **Schedulers** control the denoising process — DPM++ 2M is a good default; Euler Ancestral for variety
- **Seeds** control reproducibility — same prompt + seed = same image; vary seeds for creative exploration

## Next Steps & Resources

**Next notebook**: [04_04 — Image Editing & Inpainting](04_04_multimodal_image_editing.ipynb) — modify existing images with text guidance.

**Resources:**
- [Diffusers Documentation](https://huggingface.co/docs/diffusers/) — Official HuggingFace diffusers guide
- [Stable Diffusion Paper](https://arxiv.org/abs/2112.10752) — High-Resolution Image Synthesis with Latent Diffusion
- [CLIP Paper](https://arxiv.org/abs/2103.00020) — Learning Transferable Visual Models
- [Prompt Engineering Guide](https://huggingface.co/docs/diffusers/using-diffusers/write_own_pipeline) — Tips for better prompts
- [Civitai](https://civitai.com/) — Community models and prompt inspiration